In [4]:
# Minimal valid match statement (should compile)
from guppylang import guppy
from typing import Generic

T = guppy.type_var("T")

@guppy.enum
class EnumGen(Generic[T]):      # pyright: ignore[reportInvalidTypeForm]
    Var = {"A": T}

@guppy.enum
class Enum:
    North = {"A": int}
    South = {"B": int}
    East = {}
    West = {}

@guppy.struct
class Point(Generic[T]):         # pyright: ignore[reportInvalidTypeForm] 
    x: T                         # pyright: ignore[reportInvalidTypeForm] 
    y: int

@guppy
def main(north: Enum, point: Point[int]) -> int:
    match north:
        case Enum.North(1):
            x = 1
        case Enum.South(_):
            x = 2
        case _:
            x = 0
    
    a = Enum.North

    match a(7):
        case Enum.North(_):
            y = 3
        case _:
            y = 0
    

    match Point(3, 5):
        case Point(7, _):
            z = 4
        case _:
            z = 0
    
    match point:
        case Point(7, _):
            w = 5
        case _:
            w = 0

    match Point(EnumGen.Var(3), 5):
        case Point(EnumGen.Var(7), _):
            pass



    return 0

main.check()

In [5]:
from guppylang.decorator import guppy
from guppylang.std.builtins import owned
from guppylang.std.quantum import qubit

@guppy.struct
class Point:
    x: int
    y: int


@guppy.struct
class MyStruct:
    q1: qubit


@guppy
def foo(s1: MyStruct @owned, s2: MyStruct, p: Point) -> qubit:
    match p:
        case Point(3, 4):
            return s1.q1
        case Point(4, _):
            return s1.q1

    match s2:
        case _:
            return s1.q1

foo.check()


In [6]:
# Example: linear variable used in multiple match arms (should error if not allowed)
from guppylang import guppy
from guppylang.std.quantum import qubit, measure, owned, h

@guppy.struct
class Point:
    x: qubit
    y: int

@guppy
def fun() -> Point:
    return Point(qubit(), 4)

@guppy
def main(p: Point) -> None:
    match p:
        case Point(_, 4):
            measure(qubit())
        case _:
            pass

    h(p.x)

main.check()

In [7]:
# Example: linear variable used in multiple match arms (should error if not allowed)
from guppylang import guppy
from guppylang.std.quantum import qubit, measure, owned

@guppy.struct
class Point:
    x: qubit
    y: int

@guppy
def fun() -> Point:
    return Point(qubit(), 4)

@guppy
def main(p: Point @owned) -> None:
    match p:
        case Point(_, 4):
            pass
        case _:
            pass

    measure(p.x)

main.check()

In [46]:
# Example: linear variable used in multiple match arms (should error if not allowed)
from guppylang import guppy
from guppylang.std.quantum import qubit, measure, owned

@guppy.struct
class Point:
    x: qubit
    y: int

@guppy
def fun() -> Point:
    return Point(qubit(), 4)

@guppy
def main(p: Point @owned) -> None:
    match p:
        case Point(_, 4):
            measure(p.x)
        case _:
            measure(p.x)


main.check()

### Linearity and variables checks

In [27]:
from guppylang import guppy
from guppylang.std.quantum import qubit

@guppy.struct
class Point:
    x: qubit
    y: int

@guppy
def main(p: Point) -> None:
    match p:
        case Point(_, _):
            b = p.x
        case Point(_, 1):
            pass
        case _:
            c = p.x


main.check()

Error: Copy violation (at <In[27]>:10:9)
   | 
 8 | 
 9 | @guppy
10 | def main(p: Point) -> None:
   |          ^^^^^^^^ Borrowed argument p cannot be returned to the caller ...

Note:
   | 
12 |         case Point(_, _):
13 |             b = p.x
   |                 --- since `p.x` with non-copyable type `qubit` was already moved
   |                     here

Help: Consider writing a value back into `p.x` before returning

Guppy compilation failed due to 1 previous error


### Linearity Tests with Pattern Matching

In [30]:
# Test 2: Linear struct - should allow access after match (no extraction in pattern)
from guppylang import guppy
from guppylang.std.quantum import qubit, h

@guppy.struct
class QuantumPoint:
    q: qubit
    y: int

@guppy
def test_linear_struct_no_extract(qp: QuantumPoint) -> None:
    match qp:
        case QuantumPoint(_, 4):
            pass
        case _:
            pass
    
    # Should be OK - we didn't extract q in the pattern
    h(qp.q)

test_linear_struct_no_extract.check()

In [33]:
# Test 3: Owned linear struct - should allow access after match with owned
from guppylang import guppy
from guppylang.std.quantum import qubit, measure, owned

@guppy.struct
class QuantumPoint:
    q: qubit
    y: int

@guppy
def test_owned_linear_struct(qp: QuantumPoint @owned) -> bool:
    match qp:
        case QuantumPoint(_, 4):
            pass
        case _:
            pass
    
    # Should be OK with @owned - we can access after match
    return measure(qp.q)

test_owned_linear_struct.check()

In [36]:
# Test 4: Multiple matches on same copyable value - should work
from guppylang import guppy

@guppy.struct
class Point:
    x: qubit
    y: int

@guppy
def test_multiple_matches_copyable(p: Point) -> int:
    match p:
        case Point(_, 2):
            a = 1
        case _:
            a = 0
    
    match p:
        case Point(_, 4):
            b = 2
        case _:
            b = 0
    
    return a + b + p.y

test_multiple_matches_copyable.check()

In [39]:
# Test 6: Nested struct with mixed linear/non-linear fields
from guppylang import guppy
from guppylang.std.quantum import qubit, h

@guppy.struct
class Inner:
    value: int

@guppy.struct
class Outer:
    inner: Inner
    q: qubit

@guppy
def test_nested_struct(outer: Outer) -> None:
    match outer:
        case Outer(Inner(5), _):
            result = 1
        case Outer(_, _):
            result = 0
    
    # Should be OK - outer.inner is copyable
    h(outer.q)

test_nested_struct.check()

### Linearity Error Tests - Expected to Fail

In [61]:
@guppy.struct
class Point:
    x: qubit
    y: int

@guppy
def fun() -> Point:
    return Point(qubit(), 4)

@guppy
def describe_point(point: Point)-> Point:
    return Point(qubit(), 3)

@guppy
def main() -> None:
    # describe_point(fun())
    match describe_point(fun()):
        case _:
            pass



main.check()

In [ ]:
# TODO: NICOLa Fix the errror rendering
from guppylang import guppy
from guppylang.std.quantum import qubit

@guppy.struct
class Point:
    x: qubit
    y: int

@guppy
def main(p: Point @owned) -> None:
    match p:
        case Point(_, _):
            b = p.x


main.check()

Error: Drop violation (at <In[62]>:10:9)
   | 
 8 | 
 9 | @guppy
10 | def main(p: Point @owned) -> None:
   |          ^^^^^^^^^^^^^^^ Field `p.x` with non-droppable type `qubit` may be leaked
   |                          ...

Note:
   | 
10 | def main(p: Point @owned) -> None:
11 |     match p:
   |     --------
   | ...
13 |             b = p.x
   | ------------------- if this expression is `True`

Help: Make sure that `p.x` is consumed or returned to avoid the leak

Guppy compilation failed due to 1 previous error


In [50]:
# ERROR Test 1: different types
from guppylang import guppy
from guppylang.std.quantum import qubit, measure

@guppy.struct
class Point:
    x: int
    y: int

@guppy
def test_linear_not_consumed_all_arms(p: Point) -> int:
    
    match p:
        case Point(3, _):
            result = measure(qubit())  # q consumed here
        case Point(_,_):
            result = True
        case _:
            result = 0  # ERROR: q not consumed in this arm
    
    return result

# This should error - q is not consumed in all branches
test_linear_not_consumed_all_arms.check()

Error: Different types (at <In[50]>:21:11)
   | 
19 |             result = 0  # ERROR: q not consumed in this arm
20 | 
21 |     return result
   |            ^^^^^^ Variable `result` may refer to different types

Notes:
   | 
16 |         case Point(_,_):
17 |             result = True
   |             ------ This is of type `bool`
18 |         case _:
19 |             result = 0  # ERROR: q not consumed in this arm
   |             ------ This is of type `int`

Guppy compilation failed due to 1 previous error


In [54]:
# ERROR Test 4: Linear value created in arm not consumed
from guppylang import guppy
from guppylang.std.quantum import qubit

@guppy.struct
class Point:
    x: int
    y: int

@guppy
def test_linear_not_consumed_in_arm(p: Point) -> int:
    match p:
        case Point(3, _):
            q = qubit()  # ERROR: q created but not consumed before arm ends
            result = 1
        case _:
            result = 0
    
    return result

# This should error - q is not consumed/measured before the arm ends
test_linear_not_consumed_in_arm.check()

Error: Drop violation (at <In[54]>:14:12)
   | 
12 |     match p:
13 |         case Point(3, _):
14 |             q = qubit()  # ERROR: q created but not consumed before arm ends
   |             ^ Variable `q` with non-droppable type `qubit` is leaked

Help: Make sure that `q` is consumed or returned to avoid the leak

Guppy compilation failed due to 1 previous error
